In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'faces.zip']


In [3]:
!rm -rf /content/faces

In [4]:
!unzip -q -o "/content/drive/MyDrive/faces.zip" -d "/content/"

In [5]:
import os
print(os.listdir("/content/faces"))


['real', 'fake']


In [6]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os

In [7]:
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label_name, label_value in [("real", 0), ("fake", 1)]:
            folder = os.path.join(root_dir, label_name)
            for video_folder in os.listdir(folder):
                video_path = os.path.join(folder, video_folder)
                for img_file in os.listdir(video_path):
                    img_path = os.path.join(video_path, img_file)
                    self.samples.append((img_path, label_value))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [9]:
full_dataset = FaceDataset("/content/faces", transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Training images: 3070
Validation images: 768


In [10]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 110MB/s] 


In [11]:
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [15]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 0.3755
Epoch 2/10 - Loss: 0.1422
Epoch 3/10 - Loss: 0.0795
Epoch 4/10 - Loss: 0.0526
Epoch 5/10 - Loss: 0.0309
Epoch 6/10 - Loss: 0.0221
Epoch 7/10 - Loss: 0.0219
Epoch 8/10 - Loss: 0.0197
Epoch 9/10 - Loss: 0.0188
Epoch 10/10 - Loss: 0.0242


In [16]:
model.eval()
correct, total = 0, 0
real_correct, real_total = 0, 0
fake_correct, fake_total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        real_mask = labels == 0
        fake_mask = labels == 1
        real_total += real_mask.sum().item()
        fake_total += fake_mask.sum().item()
        real_correct += ((predicted == labels) & real_mask).sum().item()
        fake_correct += ((predicted == labels) & fake_mask).sum().item()

print(f"Overall accuracy: {100 * correct / total:.2f}%")
print(f"Real faces correct: {100 * real_correct / max(real_total, 1):.2f}%  ({real_total} images)")
print(f"Fake faces correct: {100 * fake_correct / max(fake_total, 1):.2f}%  ({fake_total} images)")

Overall accuracy: 96.48%
Real faces correct: 95.57%  (158 images)
Fake faces correct: 96.72%  (610 images)


In [17]:
import random
from torch.utils.data import Subset

random.seed(42)

def get_videos(label_name):
    folder = os.path.join("/content/faces", label_name)
    return [os.path.join(folder, v) for v in os.listdir(folder)]

real_videos = get_videos("real")
fake_videos = get_videos("fake")
random.shuffle(real_videos)
random.shuffle(fake_videos)

def split(vids, ratio=0.8):
    n = int(ratio * len(vids))
    return vids[:n], vids[n:]

real_train, real_val = split(real_videos)
fake_train, fake_val = split(fake_videos)

train_videos = set(real_train + fake_train)
val_videos = set(real_val + fake_val)

train_indices = [i for i, (p, _) in enumerate(full_dataset.samples) if os.path.dirname(p) in train_videos]
val_indices = [i for i, (p, _) in enumerate(full_dataset.samples) if os.path.dirname(p) in val_videos]

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train videos: {len(train_videos)} | Val videos: {len(val_videos)}")
print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")

Train videos: 319 | Val videos: 81
Train images: 3053 | Val images: 785


In [18]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [19]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 0.3777
Epoch 2/10 - Loss: 0.1424
Epoch 3/10 - Loss: 0.0791
Epoch 4/10 - Loss: 0.0521
Epoch 5/10 - Loss: 0.0284
Epoch 6/10 - Loss: 0.0265
Epoch 7/10 - Loss: 0.0177
Epoch 8/10 - Loss: 0.0127
Epoch 9/10 - Loss: 0.0173
Epoch 10/10 - Loss: 0.0200


In [20]:
model.eval()
correct, total = 0, 0
real_correct, real_total = 0, 0
fake_correct, fake_total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        real_mask = labels == 0
        fake_mask = labels == 1
        real_total += real_mask.sum().item()
        fake_total += fake_mask.sum().item()
        real_correct += ((predicted == labels) & real_mask).sum().item()
        fake_correct += ((predicted == labels) & fake_mask).sum().item()

print(f"Overall accuracy: {100 * correct / total:.2f}%")
print(f"Real faces correct: {100 * real_correct / max(real_total, 1):.2f}%  ({real_total} images)")
print(f"Fake faces correct: {100 * fake_correct / max(fake_total, 1):.2f}%  ({fake_total} images)")

Overall accuracy: 93.25%
Real faces correct: 82.89%  (152 images)
Fake faces correct: 95.73%  (633 images)


In [21]:
torch.save(model.state_dict(), "/content/drive/MyDrive/visual_model.pth")
print("Model saved to Google Drive.")

Model saved to Google Drive.
